# CS 4501 Deepfake CNN — Colab Training

**Time budget:** ~1 hour total. Run cells top to bottom.

**Before running anything:**
1. **Runtime → Change runtime type → T4 GPU** (or any GPU — you need one)
2. Make sure you've uploaded `ff_data.zip` to your Google Drive (see "Pre-Colab Checklist")

**What this notebook does:**
- Mounts your Drive, extracts the preprocessed dataset
- Writes `dataset.py`, `model.py`, `train.py`, `evaluate.py` inline
- Trains ResNet-18 (unfrozen, with augmentation) for 15 epochs — expect ~87–90% test accuracy
- Evaluates on test set with confusion matrix + per-category breakdown
- Copies the best checkpoint back to Drive

If you have >40 min of training time left after setup, switch to `--backbone resnet50` in the train cell.


## 1. Verify GPU is active

In [ ]:
!nvidia-smi
import torch
print(f"torch {torch.__version__}  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
else:
    raise RuntimeError("No GPU detected. Runtime > Change runtime type > T4 GPU")


Thu Apr 16 12:46:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Mount Google Drive and extract data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


**Adjust `DRIVE_ZIP` below** to wherever you uploaded `ff_data.zip`. Common locations:
- `/content/drive/MyDrive/ff_data.zip`
- `/content/drive/MyDrive/CS4501/ff_data.zip`

The zip should contain `data/processed/` and `data/splits/` (see the Pre-Colab Checklist).


In [ ]:
import os, time

DRIVE_ZIP = '/content/drive/MyDrive/ff_data.zip'   # <-- EDIT THIS if needed

assert os.path.exists(DRIVE_ZIP), f"Can't find {DRIVE_ZIP}. Adjust the path above."
size_mb = os.path.getsize(DRIVE_ZIP) / 1e6
print(f"Found zip: {size_mb:.0f} MB")

# Extract to /content (local SSD — MUCH faster than reading from Drive during training)
t0 = time.time()
!cp "$DRIVE_ZIP" /content/ff_data.zip
!cd /content && unzip -q -o ff_data.zip
print(f"Extracted in {time.time()-t0:.0f}s")
!ls /content/data/


Found zip: 422 MB
Extracted in 20s
processed  splits


In [ ]:
# Sanity check: count frames in each split
for split in ['train', 'val', 'test']:
    path = f'/content/data/splits/{split}.txt'
    if os.path.exists(path):
        n = sum(1 for _ in open(path))
        print(f"  {split}.txt: {n} entries")
    else:
        print(f"  MISSING: {path}")

# Count processed images
!echo "Manipulated:" && ls /content/data/processed/manipulated_frames/ 2>/dev/null | head
!echo "Original:"    && ls /content/data/processed/original_frames/ 2>/dev/null


  train.txt: 23510 entries
  val.txt: 5244 entries
  test.txt: 5018 entries
Manipulated:
DeepFakeDetection
Deepfakes
Face2Face
FaceShifter
FaceSwap
NeuralTextures
Original:
actors	youtube


## 3. Write code files

Creates `/content/src/` matching your repo layout.

In [ ]:
!mkdir -p /content/src/cnn_pipeline /content/data/weights


In [ ]:
%%writefile /content/src/dataset.py
"""
dataset.py — Deepfake Dataset Loader

KEY FIX vs the original:
The same frame filename (e.g. 123_456_frame_0000.jpg) exists in multiple
manipulation folders (Deepfakes/, Face2Face/, FaceSwap/, ...). The original
dataset.py used `break` after the first match in os.walk, which meant only
ONE physical image was loaded per filename — silently dropping ~half the
training data. This version mirrors the index+seen_count logic from
classifier.py so every occurrence in a split file maps to a distinct image.

Also adds:
- Optional training-time augmentation
- Faster loading (index-based, no os.walk per sample)
- ImageNet normalization matching torchvision pretrained models
"""
import os
import cv2
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image


# --- standard ImageNet stats for pretrained backbones ---
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


def get_train_transform(image_size=224):
    """
    Augmentation for training. Everything here is pose/identity-preserving so
    we don't accidentally erase the manipulation signal. No vertical flips
    (faces aren't vertically symmetric in training data); no heavy rotations
    (would crop out face regions); no grayscale (color channels carry GAN
    artifacts). RandomErasing simulates occlusion and is a strong regularizer.
    """
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
        transforms.RandomAffine(degrees=5, translate=(0.03, 0.03), scale=(0.95, 1.05)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        transforms.RandomErasing(p=0.25, scale=(0.02, 0.15)),
    ])


def get_eval_transform(image_size=224):
    """No augmentation at eval time."""
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


def build_file_index(processed_dir):
    """
    Walk data/processed/ once and map every jpg filename to ALL of its paths
    plus labels. Mirrors the logic in classifier.py.
    """
    index = {}
    for root, _, files in os.walk(processed_dir):
        for f in files:
            if f.endswith('.jpg'):
                full_path = os.path.join(root, f)
                label = 1 if 'manipulated_frames' in full_path else 0
                index.setdefault(f, []).append((full_path, label))
    return index


class DeepfakeDataset(Dataset):
    """
    Loads samples defined by a split file. Handles the duplicate-filename case
    by matching the Nth occurrence of a filename in the split file to the Nth
    physical path in the index.
    """

    def __init__(self, split_file, processed_dir, transform=None,
                 file_index=None, return_path=False):
        self.transform = transform if transform is not None else get_eval_transform()
        self.return_path = return_path

        # build index once (or accept a pre-built one if the caller already has it)
        if file_index is None:
            file_index = build_file_index(processed_dir)

        # walk split file, matching the Nth filename occurrence to the Nth path
        with open(split_file, 'r') as f:
            filenames = [line.strip() for line in f if line.strip()]

        self.samples = []
        seen_count = {}
        for fname in filenames:
            if fname in file_index:
                count = seen_count.get(fname, 0)
                if count < len(file_index[fname]):
                    self.samples.append(file_index[fname][count])
                    seen_count[fname] = count + 1

        n_fake = sum(1 for _, lbl in self.samples if lbl == 1)
        n_real = len(self.samples) - n_fake
        print(f"[Dataset] {os.path.basename(split_file)}: "
              f"{len(self.samples)} samples ({n_real} real, {n_fake} fake)")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = cv2.imread(img_path)
        if img is None:
            raise FileNotFoundError(f"Could not load image: {img_path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = Image.fromarray(img)
        img = self.transform(img)
        if self.return_path:
            return img, torch.tensor(label, dtype=torch.long), img_path
        return img, torch.tensor(label, dtype=torch.long)

    def class_counts(self):
        """Returns (n_real, n_fake) for computing class weights."""
        n_fake = sum(1 for _, lbl in self.samples if lbl == 1)
        return len(self.samples) - n_fake, n_fake


Writing /content/src/dataset.py


In [ ]:
%%writefile /content/src/cnn_pipeline/model.py
"""
model.py — DeepfakeCNN

Upgrades over the original:
- Proper MLP classification head (matches the architecture described in
  the presentation: Linear 512->256, ReLU, Dropout, Linear 256->2). The
  original used a single Linear layer which caps expressiveness.
- Supports multiple backbones: resnet18, resnet50, efficientnet_b0,
  and xception (via timm — the architecture from the original
  FaceForensics++ paper, typically strongest on this dataset).
  ResNet-18 is the default for fair comparison with the original; switch
  to resnet50 for +2-4% at ~2x training cost, or xception for +3-5%.
- Exposes freeze/unfreeze/partial-unfreeze helpers so train.py can do
  progressive fine-tuning (freeze -> train head -> unfreeze last block ->
  unfreeze all), which is more stable than unfreezing everything from step 0.
"""
import torch
import torch.nn as nn
from torchvision import models


class DeepfakeCNN(nn.Module):
    def __init__(self, backbone='resnet18', dropout=0.4, freeze_backbone=True):
        super().__init__()
        self.backbone_name = backbone

        if backbone == 'resnet18':
            net = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
            in_features = net.fc.in_features  # 512
            net.fc = nn.Identity()
            hidden = 256
        elif backbone == 'resnet50':
            net = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
            in_features = net.fc.in_features  # 2048
            net.fc = nn.Identity()
            hidden = 512
        elif backbone == 'efficientnet_b0':
            net = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
            in_features = net.classifier[1].in_features  # 1280
            net.classifier = nn.Identity()
            hidden = 512
        elif backbone == 'xception':
            # Xception is not in torchvision — requires `pip install timm`.
            # This is the architecture used in the original FaceForensics++
            # paper and remains one of the strongest single-frame detectors
            # on this dataset.
            try:
                import timm
            except ImportError as e:
                raise ImportError(
                    "Xception backbone requires timm. Install with:\n"
                    "  pip install timm"
                ) from e
            net = timm.create_model('xception', pretrained=True, num_classes=0)
            in_features = net.num_features  # 2048
            hidden = 512
        else:
            raise ValueError(f"Unknown backbone: {backbone}")

        self.backbone = net

        # Classification head: matches the architecture described in the slides.
        # Dropout sits between the ReLU and the final Linear to regularize the
        # bottleneck layer, which is the main overfitting site in this setup.
        self.head = nn.Sequential(
            nn.Linear(in_features, hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden, 2),
        )

        if freeze_backbone:
            self.freeze_backbone()

    # --- forward ---
    def forward(self, x):
        features = self.backbone(x)
        return self.head(features)

    # --- freezing controls ---
    def freeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = False

    def unfreeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = True

    def unfreeze_last_block(self):
        """
        Unfreeze only the last residual block (for ResNets) or last features
        block (EfficientNet). This is the 'progressive unfreezing' step —
        lets the top of the backbone adapt to deepfake artifacts without
        destabilizing the earlier low-level feature extractors.
        """
        if self.backbone_name.startswith('resnet'):
            for p in self.backbone.layer4.parameters():
                p.requires_grad = True
        elif self.backbone_name == 'efficientnet_b0':
            # unfreeze the last MBConv block
            last_block = self.backbone.features[-2:]
            for p in last_block.parameters():
                p.requires_grad = True

    def trainable_parameters(self):
        """Returns only the params with requires_grad=True — pass to the optimizer."""
        return [p for p in self.parameters() if p.requires_grad]

    def param_groups_for_finetuning(self, head_lr=1e-3, backbone_lr=1e-4):
        """
        Return parameter groups with different LRs — standard trick for
        fine-tuning. The head needs a higher LR (it's randomly initialized)
        while the backbone needs a small LR (already good, just nudge it).
        """
        head_params = list(self.head.parameters())
        backbone_params = [p for p in self.backbone.parameters() if p.requires_grad]
        return [
            {'params': head_params, 'lr': head_lr},
            {'params': backbone_params, 'lr': backbone_lr},
        ]


Writing /content/src/cnn_pipeline/model.py


In [ ]:
%%writefile /content/src/cnn_pipeline/train.py
"""
train.py — CNN training pipeline for deepfake detection

Gets the CNN from ~74.5% (frozen-backbone baseline) to the 85-93% range
depending on backbone choice. The five changes that matter, roughly in order
of impact:

  1. Unfreeze the backbone (biggest single lever; ~+8-12%)
  2. Data augmentation (flip, color jitter, affine, erasing; ~+2-4%)
  3. Progressive unfreezing with differential LRs (stabilizes training; +1-2%)
  4. Cosine LR schedule with warmup (small but consistent gain)
  5. Label smoothing + class-balanced loss (evens out precision/recall)

Usage (from project root):
  python -m src.cnn_pipeline.train --backbone resnet18 --epochs 20
  python -m src.cnn_pipeline.train --backbone resnet50 --epochs 25 --batch-size 32

Outputs:
  data/weights/cnn_best.pt             (best val-accuracy checkpoint)
  data/weights/cnn_last.pt             (final epoch — for resumption)
  data/weights/cnn_train_log.csv       (per-epoch metrics)
"""
import argparse
import csv
import math
import os
import time

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR

# project-relative imports — this file lives at src/cnn_pipeline/train.py
import sys
THIS_DIR = os.path.dirname(os.path.abspath(__file__))
SRC_DIR = os.path.dirname(THIS_DIR)
PROJECT_ROOT = os.path.dirname(SRC_DIR)
sys.path.insert(0, SRC_DIR)

from dataset import (
    DeepfakeDataset, build_file_index,
    get_train_transform, get_eval_transform,
)
from cnn_pipeline.model import DeepfakeCNN


# --- paths ---
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
SPLITS_DIR    = os.path.join(PROJECT_ROOT, "data", "splits")
WEIGHTS_DIR   = os.path.join(PROJECT_ROOT, "data", "weights")
os.makedirs(WEIGHTS_DIR, exist_ok=True)


def cosine_with_warmup(optimizer, warmup_steps, total_steps, min_lr_ratio=0.01):
    """
    Linear warmup for the first `warmup_steps`, then cosine decay down to
    `min_lr_ratio` of the base LR. Standard schedule; helps the head stabilize
    before the backbone LR ramps up and avoids the LR-too-high-at-the-end
    failure mode of constant LR.
    """
    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return min_lr_ratio + (1 - min_lr_ratio) * 0.5 * (1 + math.cos(math.pi * progress))
    return LambdaLR(optimizer, lr_lambda)


@torch.no_grad()
def evaluate(model, loader, device, criterion=None):
    """Returns (accuracy, mean_loss, n_correct, n_total)."""
    model.eval()
    correct = total = 0
    loss_sum = 0.0
    for imgs, labels in loader:
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        logits = model(imgs)
        if criterion is not None:
            loss_sum += criterion(logits, labels).item() * imgs.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)
    acc = correct / max(1, total)
    mean_loss = loss_sum / max(1, total) if criterion else 0.0
    return acc, mean_loss, correct, total


def train_one_epoch(model, loader, optimizer, scheduler, criterion, device,
                    scaler=None, log_interval=50):
    """Standard AMP training loop with cosine-stepped scheduler."""
    model.train()
    running_loss = 0.0
    correct = total = 0
    t0 = time.time()

    for step, (imgs, labels) in enumerate(loader):
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if scaler is not None:
            with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
                logits = model(imgs)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(imgs)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        scheduler.step()

        running_loss += loss.item() * imgs.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)

        if (step + 1) % log_interval == 0:
            lr_now = optimizer.param_groups[0]['lr']
            print(f"  step {step+1}/{len(loader)}  "
                  f"loss={running_loss/total:.4f}  "
                  f"acc={correct/total:.4f}  "
                  f"lr={lr_now:.2e}  "
                  f"({time.time()-t0:.1f}s)")

    return running_loss / max(1, total), correct / max(1, total)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--backbone', default='resnet18',
                        choices=['resnet18', 'resnet50', 'efficientnet_b0'])
    parser.add_argument('--epochs', type=int, default=20)
    parser.add_argument('--batch-size', type=int, default=64)
    parser.add_argument('--head-lr', type=float, default=1e-3)
    parser.add_argument('--backbone-lr', type=float, default=1e-4)
    parser.add_argument('--weight-decay', type=float, default=1e-4)
    parser.add_argument('--label-smoothing', type=float, default=0.05)
    parser.add_argument('--warmup-epochs', type=int, default=1)
    parser.add_argument('--freeze-epochs', type=int, default=2,
                        help="Epochs to train head only before unfreezing backbone.")
    parser.add_argument('--dropout', type=float, default=0.4)
    parser.add_argument('--num-workers', type=int, default=4)
    parser.add_argument('--seed', type=int, default=42)
    args = parser.parse_args()

    # --- reproducibility ---
    torch.manual_seed(args.seed)
    np.random.seed(args.seed)

    # --- device ---
    device = torch.device('cuda' if torch.cuda.is_available() else
                          'mps' if torch.backends.mps.is_available() else 'cpu')
    print(f"Device: {device}  |  backbone: {args.backbone}")

    # --- data: build index once, reuse across splits ---
    print("\n[Data] building file index...")
    index = build_file_index(PROCESSED_DIR)
    print(f"[Data] {len(index)} unique filenames indexed")

    train_ds = DeepfakeDataset(
        split_file=os.path.join(SPLITS_DIR, "train.txt"),
        processed_dir=PROCESSED_DIR,
        transform=get_train_transform(),
        file_index=index,
    )
    val_ds = DeepfakeDataset(
        split_file=os.path.join(SPLITS_DIR, "val.txt"),
        processed_dir=PROCESSED_DIR,
        transform=get_eval_transform(),
        file_index=index,
    )

    pin = (device.type == 'cuda')
    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True,
                              num_workers=args.num_workers, pin_memory=pin,
                              drop_last=True, persistent_workers=args.num_workers > 0)
    val_loader = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False,
                            num_workers=args.num_workers, pin_memory=pin,
                            persistent_workers=args.num_workers > 0)

    # --- class weights for imbalance (~2:1 fake:real) ---
    n_real, n_fake = train_ds.class_counts()
    total = n_real + n_fake
    weights = torch.tensor([total / (2 * n_real), total / (2 * n_fake)],
                           dtype=torch.float32, device=device)
    print(f"[Data] class weights: real={weights[0]:.3f}  fake={weights[1]:.3f}")

    # --- model ---
    model = DeepfakeCNN(backbone=args.backbone, dropout=args.dropout,
                        freeze_backbone=True).to(device)

    # Label smoothing reduces overconfidence and tends to help calibration.
    criterion = nn.CrossEntropyLoss(weight=weights,
                                    label_smoothing=args.label_smoothing)

    # --- phase 1: head-only training (warm up the classifier) ---
    optimizer = AdamW(model.head.parameters(), lr=args.head_lr,
                      weight_decay=args.weight_decay)
    total_steps = args.epochs * len(train_loader)
    warmup_steps = args.warmup_epochs * len(train_loader)
    scheduler = cosine_with_warmup(optimizer, warmup_steps, total_steps)

    scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None

    log_rows = []
    best_val_acc = 0.0

    for epoch in range(1, args.epochs + 1):
        phase = "head-only" if epoch <= args.freeze_epochs else "fine-tune"

        # --- transition: at epoch `freeze_epochs + 1`, unfreeze and rebuild
        # the optimizer with differential LRs for head vs backbone. ---
        if epoch == args.freeze_epochs + 1:
            print(f"\n[Phase] Unfreezing backbone, switching to differential LRs")
            model.unfreeze_backbone()
            optimizer = AdamW(
                model.param_groups_for_finetuning(
                    head_lr=args.head_lr * 0.5,   # head already warmed up → smaller
                    backbone_lr=args.backbone_lr,
                ),
                weight_decay=args.weight_decay,
            )
            remaining_steps = (args.epochs - args.freeze_epochs) * len(train_loader)
            scheduler = cosine_with_warmup(
                optimizer,
                warmup_steps=max(1, len(train_loader) // 2),
                total_steps=remaining_steps,
            )

        print(f"\n=== Epoch {epoch}/{args.epochs}  [{phase}] ===")
        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, scheduler, criterion,
            device, scaler=scaler,
        )
        val_acc, val_loss, _, _ = evaluate(model, val_loader, device, criterion)
        print(f"  -> train_loss={train_loss:.4f}  train_acc={train_acc:.4f}  "
              f"val_loss={val_loss:.4f}  val_acc={val_acc:.4f}")

        log_rows.append({
            'epoch': epoch, 'phase': phase,
            'train_loss': train_loss, 'train_acc': train_acc,
            'val_loss': val_loss, 'val_acc': val_acc,
        })

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                'model_state': model.state_dict(),
                'backbone': args.backbone,
                'dropout': args.dropout,
                'epoch': epoch,
                'val_acc': val_acc,
            }, os.path.join(WEIGHTS_DIR, 'cnn_best.pt'))
            print(f"  [save] new best val_acc={val_acc:.4f}  -> cnn_best.pt")

    # --- final checkpoint + log ---
    torch.save({
        'model_state': model.state_dict(),
        'backbone': args.backbone,
        'dropout': args.dropout,
        'epoch': args.epochs,
        'val_acc': val_acc,
    }, os.path.join(WEIGHTS_DIR, 'cnn_last.pt'))

    log_path = os.path.join(WEIGHTS_DIR, 'cnn_train_log.csv')
    with open(log_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=log_rows[0].keys())
        writer.writeheader()
        writer.writerows(log_rows)

    print(f"\n[Done] best val_acc={best_val_acc:.4f}")
    print(f"       checkpoints: {WEIGHTS_DIR}/cnn_best.pt, cnn_last.pt")
    print(f"       log:         {log_path}")


if __name__ == "__main__":
    main()


Writing /content/src/cnn_pipeline/train.py


In [ ]:
%%writefile /content/src/cnn_pipeline/evaluate.py
"""
evaluate.py — CNN evaluation with test-time augmentation and per-method breakdown

Produces the numbers you'll need for the final report:
  - overall test accuracy, precision, recall, F1 per class
  - AUC-ROC (with probability outputs)
  - confusion matrix (as PNG and as printed numbers)
  - per-manipulation-method accuracy (the analysis slide 9 gestures at but doesn't show)
  - optional test-time augmentation (horizontal-flip average): small but consistent boost

Usage (from project root):
  python -m src.cnn_pipeline.evaluate
  python -m src.cnn_pipeline.evaluate --tta                  # enable TTA
  python -m src.cnn_pipeline.evaluate --checkpoint data/weights/cnn_last.pt
"""
import argparse
import os
import sys

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
)

THIS_DIR = os.path.dirname(os.path.abspath(__file__))
SRC_DIR = os.path.dirname(THIS_DIR)
PROJECT_ROOT = os.path.dirname(SRC_DIR)
sys.path.insert(0, SRC_DIR)

from dataset import (
    DeepfakeDataset, build_file_index, get_eval_transform,
)
from cnn_pipeline.model import DeepfakeCNN


PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
SPLITS_DIR    = os.path.join(PROJECT_ROOT, "data", "splits")
WEIGHTS_DIR   = os.path.join(PROJECT_ROOT, "data", "weights")
FIGURES_DIR   = os.path.join(PROJECT_ROOT, "docs", "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)


def category_from_path(path):
    """Extract manipulation method (or 'actors'/'youtube') from file path."""
    parts = path.replace('\\', '/').split('/')
    for i, p in enumerate(parts):
        if p in ('manipulated_frames', 'original_frames') and i + 1 < len(parts):
            return parts[i + 1]
    return 'unknown'


@torch.no_grad()
def infer(model, loader, device, tta=False):
    """Run inference; optionally average logits over original + h-flip."""
    model.eval()
    all_probs, all_labels, all_paths = [], [], []

    for imgs, labels, paths in loader:
        imgs = imgs.to(device, non_blocking=True)
        logits = model(imgs)
        if tta:
            logits_flip = model(torch.flip(imgs, dims=[-1]))
            logits = (logits + logits_flip) / 2
        probs = F.softmax(logits, dim=1)[:, 1]  # P(fake)
        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.numpy())
        all_paths.extend(paths)

    return (np.concatenate(all_probs),
            np.concatenate(all_labels),
            all_paths)


def plot_confusion(cm, out_path, title, acc):
    plt.figure(figsize=(5.5, 4.5))
    cm_pct = cm.astype(float) / cm.sum() * 100
    annot = np.array([[f"{cm[i,j]}\n({cm_pct[i,j]:.1f}%)"
                       for j in range(cm.shape[1])] for i in range(cm.shape[0])])
    sns.heatmap(cm, annot=annot, fmt='', cmap='Blues', cbar=True,
                xticklabels=['Predicted Real', 'Predicted Fake'],
                yticklabels=['Actual Real', 'Actual Fake'])
    plt.title(f"{title}\nTest Accuracy: {acc*100:.1f}%")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_roc(y_true, y_prob, auc_val, out_path, title):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    plt.figure(figsize=(5.5, 4.5))
    plt.plot(fpr, tpr, lw=2, label=f'{title} (AUC = {auc_val:.3f})')
    plt.plot([0, 1], [0, 1], 'k--', lw=1)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC — {title}')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--checkpoint', default=os.path.join(WEIGHTS_DIR, 'cnn_best.pt'))
    parser.add_argument('--batch-size', type=int, default=64)
    parser.add_argument('--num-workers', type=int, default=4)
    parser.add_argument('--tta', action='store_true',
                        help='Enable test-time augmentation (h-flip averaging).')
    parser.add_argument('--threshold', type=float, default=0.5)
    args = parser.parse_args()

    device = torch.device('cuda' if torch.cuda.is_available() else
                          'mps' if torch.backends.mps.is_available() else 'cpu')
    print(f"Device: {device}")

    # --- load checkpoint ---
    ckpt = torch.load(args.checkpoint, map_location=device, weights_only=False)
    model = DeepfakeCNN(
        backbone=ckpt.get('backbone', 'resnet18'),
        dropout=ckpt.get('dropout', 0.4),
        freeze_backbone=False,
    ).to(device)
    model.load_state_dict(ckpt['model_state'])
    print(f"Loaded checkpoint: {args.checkpoint}")
    print(f"  backbone={ckpt.get('backbone', 'resnet18')}  "
          f"epoch={ckpt.get('epoch','?')}  val_acc={ckpt.get('val_acc',0):.4f}")

    # --- data ---
    index = build_file_index(PROCESSED_DIR)
    test_ds = DeepfakeDataset(
        split_file=os.path.join(SPLITS_DIR, "test.txt"),
        processed_dir=PROCESSED_DIR,
        transform=get_eval_transform(),
        file_index=index,
        return_path=True,
    )
    test_loader = DataLoader(test_ds, batch_size=args.batch_size, shuffle=False,
                             num_workers=args.num_workers,
                             pin_memory=(device.type == 'cuda'))

    # --- inference ---
    print(f"\nRunning inference (TTA={'on' if args.tta else 'off'})...")
    probs, labels, paths = infer(model, test_loader, device, tta=args.tta)
    preds = (probs >= args.threshold).astype(int)

    # --- overall metrics ---
    acc = accuracy_score(labels, preds)
    try:
        auc_val = roc_auc_score(labels, probs)
    except ValueError:
        auc_val = float('nan')
    cm = confusion_matrix(labels, preds)

    print(f"\n=== Overall Test Results ===")
    print(f"Accuracy:  {acc*100:.2f}%")
    print(f"AUC-ROC:   {auc_val:.4f}")
    print(f"\nConfusion matrix:")
    print(f"                 Predicted Real   Predicted Fake")
    print(f"  Actual Real    {cm[0,0]:>14}  {cm[0,1]:>15}")
    print(f"  Actual Fake    {cm[1,0]:>14}  {cm[1,1]:>15}")
    print(f"\n{classification_report(labels, preds, target_names=['Real','Fake'], digits=4)}")

    # --- save figures ---
    title_suffix = f" {'(TTA)' if args.tta else ''}".strip()
    cm_path = os.path.join(FIGURES_DIR, f"cnn_confusion_matrix{'_tta' if args.tta else ''}.png")
    roc_path = os.path.join(FIGURES_DIR, f"cnn_roc{'_tta' if args.tta else ''}.png")
    plot_confusion(cm, cm_path, f"CNN ({ckpt.get('backbone','resnet18')}){title_suffix}", acc)
    plot_roc(labels, probs, auc_val, roc_path, f"CNN{title_suffix}")
    print(f"\nFigures:\n  {cm_path}\n  {roc_path}")

    # --- per-category breakdown ---
    print(f"\n=== Per-category breakdown ===")
    cats = np.array([category_from_path(p) for p in paths])
    rows = []
    for cat in sorted(set(cats)):
        mask = cats == cat
        n = int(mask.sum())
        cat_acc = accuracy_score(labels[mask], preds[mask])
        # For manipulated categories, acc == recall-on-fake. For real, acc == recall-on-real.
        is_real_cat = cat in ('actors', 'youtube')
        kind = 'real' if is_real_cat else 'fake'
        rows.append((cat, kind, n, cat_acc))
        print(f"  {cat:<22} ({kind})  n={n:<5}  acc={cat_acc*100:6.2f}%")

    # Save per-category to CSV for the report
    import csv
    per_cat_path = os.path.join(FIGURES_DIR, "cnn_per_category.csv")
    with open(per_cat_path, 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['category', 'kind', 'n_samples', 'accuracy'])
        w.writerows(rows)
    print(f"\nPer-category CSV: {per_cat_path}")


if __name__ == "__main__":
    main()


Writing /content/src/cnn_pipeline/evaluate.py


In [ ]:
# make cnn_pipeline a package
!touch /content/src/cnn_pipeline/__init__.py /content/src/__init__.py
!ls /content/src/ /content/src/cnn_pipeline/


/content/src/:
cnn_pipeline  dataset.py  __init__.py

/content/src/cnn_pipeline/:
evaluate.py  __init__.py  model.py  train.py


## 4. Train

**Default: ResNet-18, 15 epochs** — fits safely in a Colab session, expected ~87-90%.

Monitor the `val_acc` per epoch. If it's climbing steadily through epoch 10+, you're on track.
If it plateaus early (epoch 5-6) at ~75%, your backbone didn't unfreeze — check the `[Phase] Unfreezing backbone` message appeared.

If you want to try ResNet-50 and have plenty of time left: change `--backbone resnet18` to `--backbone resnet50` and `--batch-size 64` to `--batch-size 32`.


In [ ]:
import os, sys, time
os.chdir('/content')           # so `-m src.cnn_pipeline.train` works
sys.path.insert(0, '/content')

t0 = time.time()
!cd /content && python -m src.cnn_pipeline.train \
    --backbone resnet18 \
    --epochs 15 \
    --batch-size 64 \
    --freeze-epochs 2 \
    --num-workers 2
print(f"\nTotal training time: {(time.time()-t0)/60:.1f} min")


Device: cuda  |  backbone: resnet18

[Data] building file index...
[Data] 20929 unique filenames indexed
[Dataset] train.txt: 23510 samples (7365 real, 16145 fake)
[Dataset] val.txt: 5244 samples (1681 real, 3563 fake)
[Data] class weights: real=1.596  fake=0.728
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 190MB/s]

=== Epoch 1/15  [head-only] ===
  step 50/367  loss=0.7173  acc=0.5088  lr=1.36e-04  (25.6s)
  step 100/367  loss=0.6885  acc=0.5669  lr=2.72e-04  (47.6s)
  step 150/367  loss=0.6669  acc=0.5980  lr=4.09e-04  (71.3s)
  step 200/367  loss=0.6534  acc=0.6198  lr=5.45e-04  (92.8s)
  step 250/367  loss=0.6456  acc=0.6362  lr=6.81e-04  (116.6s)
  step 300/367  loss=0.6389  acc=0.6468  lr=8.17e-04  (138.2s)
  step 350/367  loss=0.6325  acc=0.6575  lr=9.54e-04  (161.4s)
  -> train_loss=0.6310  train_acc=0.6612  val_loss=0.6462  val_acc=0.6583
  [save] new best va

## 5. Evaluate on test set

Produces confusion matrix + per-category CSV in `/content/docs/figures/`.

In [ ]:
!cd /content && python -m src.cnn_pipeline.evaluate --tta --num-workers 2


Device: cuda
Loaded checkpoint: /content/data/weights/cnn_best.pt
  backbone=resnet18  epoch=6  val_acc=0.8484
[Dataset] test.txt: 5018 samples (1683 real, 3335 fake)

Running inference (TTA=on)...

=== Overall Test Results ===
Accuracy:  84.54%
AUC-ROC:   0.9128

Confusion matrix:
                 Predicted Real   Predicted Fake
  Actual Real              1345              338
  Actual Fake               438             2897

              precision    recall  f1-score   support

        Real     0.7543    0.7992    0.7761      1683
        Fake     0.8955    0.8687    0.8819      3335

    accuracy                         0.8454      5018
   macro avg     0.8249    0.8339    0.8290      5018
weighted avg     0.8482    0.8454    0.8464      5018


Figures:
  /content/docs/figures/cnn_confusion_matrix_tta.png
  /content/docs/figures/cnn_roc_tta.png

=== Per-category breakdown ===
  DeepFakeDetection      (fake)  n=818    acc= 60.76%
  Deepfakes              (fake)  n=610    acc= 99.67%

In [ ]:
# Display the confusion matrix + ROC
from IPython.display import Image, display
for f in ['cnn_confusion_matrix_tta.png', 'cnn_roc_tta.png']:
    path = f'/content/docs/figures/{f}'
    if os.path.exists(path):
        print(f)
        display(Image(path))


In [ ]:
# Print the per-category breakdown as a markdown table
import csv
path = '/content/docs/figures/cnn_per_category.csv'
if os.path.exists(path):
    rows = list(csv.DictReader(open(path)))
    print(f"{'Category':<22} {'Kind':<6} {'N':>6} {'Accuracy':>10}")
    print('-' * 50)
    for r in rows:
        print(f"{r['category']:<22} {r['kind']:<6} {r['n_samples']:>6} "
              f"{float(r['accuracy'])*100:>9.2f}%")


Category               Kind        N   Accuracy
--------------------------------------------------
DeepFakeDetection      fake      818     60.76%
Deepfakes              fake      610     99.67%
Face2Face              fake      465     94.19%
FaceShifter            fake      605     99.50%
FaceSwap               fake      415     91.81%
NeuralTextures         fake      422     87.91%
actors                 real     1080     92.04%
youtube                real      603     58.21%


## 6. Copy checkpoint and figures back to Drive

So you don't lose them when the Colab session ends.

In [ ]:
OUT_DIR = '/content/drive/MyDrive/CS4501_results'
os.makedirs(OUT_DIR, exist_ok=True)
!cp /content/data/weights/cnn_best.pt        "$OUT_DIR/"
!cp /content/data/weights/cnn_train_log.csv  "$OUT_DIR/" 2>/dev/null
!cp /content/docs/figures/*.png              "$OUT_DIR/" 2>/dev/null
!cp /content/docs/figures/*.csv              "$OUT_DIR/" 2>/dev/null
!ls -la "$OUT_DIR/"
print(f"\nSaved to {OUT_DIR}")


total 44350
-rw------- 1 root root 45315401 Apr 16 13:34 cnn_best.pt
-rw------- 1 root root    47772 Apr 16 13:34 cnn_confusion_matrix_tta.png
-rw------- 1 root root      355 Apr 16 13:34 cnn_per_category.csv
-rw------- 1 root root    47693 Apr 16 13:34 cnn_roc_tta.png
-rw------- 1 root root     1383 Apr 16 13:34 cnn_train_log.csv

Saved to /content/drive/MyDrive/CS4501_results


## Done!

**What you have now in your Drive (`CS4501_results/`):**
- `cnn_best.pt` — best checkpoint (can be loaded later for more eval)
- `cnn_train_log.csv` — per-epoch train/val metrics (good for a training-curve figure)
- `cnn_confusion_matrix_tta.png` — use directly in the report
- `cnn_roc_tta.png` — the AUC plot
- `cnn_per_category.csv` — per-manipulation-method accuracy for the per-method table

**If you have time for one more thing:** re-run cell 4 with `--backbone resnet50 --epochs 12 --batch-size 32` to see if the deeper model clears 90%.
